Library Import

In [1]:
from keras.models import Sequential
from keras.layers import Dense, Dropout, LSTM
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import tensorflow as tf

2024-08-24 15:39:27.267695: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-08-24 15:39:27.267810: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-08-24 15:39:27.399445: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Loading Data

In [2]:
train_data = pd.read_csv("/kaggle/input/competition-dataset/train_data.csv")
new_data = pd.read_csv("/kaggle/input/competition-dataset/systems_new.csv")
test_data = pd.read_csv("/kaggle/input/competition-dataset/test_data_masked.csv")


merged_data = pd.merge(train_data,new_data)
merged_test_data = pd.merge(test_data,new_data)

merged_data


,system_id,timestamp,generation_W,load_W,connection_type,location,panels_capacity,load_capacity
0,17,2023-08-14 01:50:00,0.00000,525.66667,COMMERCIAL,MARDAN,5.005,5.5
1,17,2023-08-14 02:00:00,0.00000,518.90000,COMMERCIAL,MARDAN,5.005,5.5
2,17,2023-08-14 02:10:00,0.00000,528.50001,COMMERCIAL,MARDAN,5.005,5.5
3,17,2023-08-14 02:20:00,0.00000,517.99999,COMMERCIAL,MARDAN,5.005,5.5
4,17,2023-08-14 02:30:00,0.00000,520.90001,COMMERCIAL,MARDAN,5.005,5.5
...,...,...,...,...,...,...,...,...
3845823,57,2024-08-09 07:30:00,2969.58337,8353.03565,COMMERCIAL,KARACHI,19.800,20.0
3845824,57,2024-08-09 07:40:00,3510.99999,8843.46429,COMMERCIAL,KARACHI,19.800,20.0
3845825,57,2024-08-09 07:50:00,3689.99999,8262.64100,COMMERCIAL,KARACHI,19.800,20.0
3845826,57,2024-08-09 08:00:00,3868.62502,10244.71429,COMMERCIAL,KARACHI,19.800,20.0


Splitting Timestamp

In [3]:

# Train Data
merged_data["timestamp"] = merged_data["timestamp"].astype("datetime64[ns]")

merged_data['year'] = merged_data['timestamp'].dt.year
merged_data['month'] = merged_data['timestamp'].dt.month
merged_data['day'] = merged_data['timestamp'].dt.day
merged_data['hour'] = merged_data['timestamp'].dt.hour
merged_data['minute'] = merged_data['timestamp'].dt.minute

merged_data = merged_data.drop(["timestamp"],axis=1)


# Test Data
merged_test_data["timestamp"] = merged_test_data["timestamp"].astype("datetime64[ns]")

merged_test_data['year'] = merged_test_data['timestamp'].dt.year
merged_test_data['month'] = merged_test_data['timestamp'].dt.month
merged_test_data['day'] = merged_test_data['timestamp'].dt.day
merged_test_data['hour'] = merged_test_data['timestamp'].dt.hour
merged_test_data['minute'] = merged_test_data['timestamp'].dt.minute

merged_test_data = merged_test_data.drop(["timestamp"], axis=1)

Manual Encoding

In [4]:

values = set(list(merged_data["location"].unique()) + list(merged_test_data["location"].unique()))

dic = {}
count = 0

for i in values:
    dic.update({i:count})
    count += 1


merged_data["location"] = merged_data["location"].map(dic)
merged_data["connection_type"] = merged_data["connection_type"].map({"COMMERCIAL":0,"RESIDENTIAL":1})

merged_test_data["location"] = merged_test_data["location"].map(dic)
merged_test_data["connection_type"] = merged_test_data["connection_type"].map({"COMMERCIAL":0,"RESIDENTIAL":1})


val_data = merged_test_data[(merged_test_data['load_W'] != -1) & (merged_test_data['load_W'] != -2)]

merged_test_data = merged_test_data[(merged_test_data['load_W'] == -1)]


merged_data = pd.merge(merged_data, val_data, how = "outer")

merged_data

,system_id,generation_W,load_W,connection_type,location,panels_capacity,load_capacity,year,month,day,hour,minute,test_id
0,1,0.00000,0.00000,1,2,11.125,10.0,2024,1,31,0,0,2JQDQNJM
1,1,0.00000,0.00000,1,2,11.125,10.0,2024,6,22,1,50,5T9TBC1T
2,1,0.00000,56.70000,1,2,11.125,10.0,2024,3,28,5,30,VH02RKIJ
3,1,0.00000,56.88889,1,2,11.125,10.0,2023,11,12,6,10,782S5PBI
4,1,0.00000,60.30000,1,2,11.125,10.0,2024,3,28,5,20,A4RT4R3S
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4314160,107,10060.19993,1032.08333,1,2,12.285,10.0,2024,5,2,12,10,8EEXFUH0
4314161,107,10099.49999,3059.09092,1,2,12.285,10.0,2023,9,22,11,40,L21L5GL7
4314162,107,10284.99997,849.33334,1,2,12.285,10.0,2024,4,20,12,20,ZCDUHMHS
4314163,107,10298.00014,766.07142,1,2,12.285,10.0,2024,3,31,12,20,86KKUP8N


Adding Features

In [ ]:

# Train Data
merged_data["Load Utilization"] = (merged_data["load_W"] / merged_data["load_capacity"]) * 100

merged_data["Generation Efficiency"] = (merged_data["generation_W"] / merged_data["panels_capacity"]) * 100

merged_data["Capacity Ratio"] = (merged_data["panels_capacity"] / merged_data["load_capacity"])

merged_data["Load per panel"] = (merged_data["load_W"] / merged_data["panels_capacity"])

merged_data["Generation per panel"] = (merged_data["generation_W"] / merged_data["panels_capacity"])

# Test Data

merged_test_data["Load Utilization"] = (merged_test_data["load_W"] / merged_test_data["load_capacity"]) * 100

merged_test_data["Generation Efficiency"] = (merged_test_data["generation_W"] / merged_test_data["panels_capacity"]) * 100

merged_test_data["Capacity Ratio"] = (merged_test_data["panels_capacity"] / merged_test_data["load_capacity"])

merged_test_data["Load per panel"] = (merged_test_data["load_W"] / merged_test_data["panels_capacity"])

merged_test_data["Generation per panel"] = (merged_test_data["generation_W"] / merged_test_data["panels_capacity"])



merged_data

In [5]:
X_train = merged_data.drop(["load_W","generation_W","test_id"],axis=1)
# X = tf.expand_dims(X, axis=1)
Y_train = merged_data[["generation_W","load_W"]]

X_test = merged_test_data.drop(["load_W","generation_W","test_id"],axis=1)
# X = tf.expand_dims(X, axis=1)
Y_test = merged_test_data[["generation_W","load_W"]]

X_val = val_data.drop(["load_W","generation_W","test_id"],axis=1)
# X = tf.expand_dims(X, axis=1)
Y_val = val_data[["generation_W","load_W"]]


In [6]:
# X_train_column_names = X_train.columns
# X_test_column_names = X_test.columns

# scaler = MinMaxScaler()

# X_train = scaler.fit_transform(X_train)
# X_train = pd.DataFrame(X_train, columns= X_train_column_names)

# X_test = scaler.fit_transform(X_test)
# X_test = pd.DataFrame(X_test, columns = X_test_column_names)


In [7]:
regressor = Sequential()

# Adding the first LSTM layer and some Dropout regularisation
regressor.add(LSTM(units = 50, return_sequences = True, input_shape = (X_train.shape[1], 1)))
regressor.add(Dropout(0.2))

# Adding a second LSTM layer and some Dropout regularisation
regressor.add(LSTM(units = 50, return_sequences = True))
regressor.add(Dropout(0.2))

# Adding a third LSTM layer and some Dropout regularisation
regressor.add(LSTM(units = 50, return_sequences = True))
regressor.add(Dropout(0.2))

# Adding a fourth LSTM layer and some Dropout regularisation
regressor.add(LSTM(units = 50))
regressor.add(Dropout(0.2))

# Adding the output layer
regressor.add(Dense(units = 2))

# Compiling the RNN
regressor.compile(optimizer = 'adam', loss = 'mae')

/opt/conda/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
regressor.fit(X_train, Y_train, epochs=50, batch_size=128, validation_data = (X_val,Y_val))

Epoch 1/50
33705/33705 ━━━━━━━━━━━━━━━━━━━━ 305s 9ms/step - loss: 1508.2928 - val_loss: 1081.5463
Epoch 2/50
33705/33705 ━━━━━━━━━━━━━━━━━━━━ 297s 9ms/step - loss: 1271.4752 - val_loss: 871.5696
Epoch 3/50
33705/33705 ━━━━━━━━━━━━━━━━━━━━ 297s 9ms/step - loss: 1053.1776 - val_loss: 698.5443
Epoch 4/50
33705/33705 ━━━━━━━━━━━━━━━━━━━━ 297s 9ms/step - loss: 894.5742 - val_loss: 585.8920
Epoch 5/50
33705/33705 ━━━━━━━━━━━━━━━━━━━━ 299s 9ms/step - loss: 782.0610 - val_loss: 514.5684
Epoch 6/50
33705/33705 ━━━━━━━━━━━━━━━━━━━━ 307s 9ms/step - loss: 707.7206 - val_loss: 468.2610
Epoch 7/50
33705/33705 ━━━━━━━━━━━━━━━━━━━━ 310s 9ms/step - loss: 660.2466 - val_loss: 439.7099
Epoch 8/50
33705/33705 ━━━━━━━━━━━━━━━━━━━━ 310s 9ms/step - loss: 629.7651 - val_loss: 424.2845
Epoch 9/50
33705/33705 ━━━━━━━━━━━━━━━━━━━━ 320s 9ms/step - loss: 607.4664 - val_loss: 411.3943
Epoch 10/50
33705/33705 ━━━━━━━━━━━━━━━━━━━━ 311s 9ms/step - loss: 592.5364 - val_loss: 405.2572
Epoch 11/50
33705/33705 ━━━━━━━━━━━

In [ ]:
predict = regressor.predict(X_test)

In [ ]:
x = pd.DataFrame(predict)
x = x.clip(lower = 0)
x

In [ ]:
# merged_test_data = merged_test_data.drop(["connection_type","location","panels_capacity","load_capacity"],axis=1)
# merged_test_data['timestamp'] = pd.to_datetime({
#     'year': merged_test_data['year'],
#     'month': merged_test_data['month'],
#     'day': merged_test_data['day'],
#     'hour': merged_test_data['hour'],
#     'minute': merged_test_data['minute']
# })

# merged_test_data = merged_test_data.drop(['year', 'month', 'day', 'hour', 'minute'], axis=1)

merged_test_data["generation_W"] = x[0].values
merged_test_data["load_W"] = x[1].values



In [ ]:
merged_test_data = merged_test_data[["test_id","system_id","timestamp","generation_W","load_W"]]

merged_test_data.to_csv("LSTM_1.csv",index=False)

merged_test_data